# Docker Network Types

Every container runs in its own network namespace — it has its own network interfaces, IP address, and routing table. Docker connects containers to each other and to the outside world through **networks**.

Docker ships with several built-in network drivers, each solving a different connectivity problem. Choosing the right one is the difference between a secure, well-isolated deployment and one where everything can talk to everything.

Topics:
- The five built-in network drivers: bridge, host, none, overlay, macvlan
- Default bridge network and its limitations
- `host` network — when to skip container networking entirely
- `none` — total network isolation
- `overlay` — multi-host networking in Swarm
- `macvlan` — containers with their own MAC address
- Inspecting networks
- Which driver to use when

## 1. How Container Networking Works

When Docker creates a network, it creates a virtual network interface on the host (a Linux bridge, or a veth pair). Each container on that network gets a virtual ethernet interface connected to it.

```
Host
├── eth0  (physical NIC, e.g. 192.168.1.100)
└── docker0  (Docker bridge, e.g. 172.17.0.1)
    ├── veth0abc  ──► container1 eth0 (172.17.0.2)
    └── veth1def  ──► container2 eth0 (172.17.0.3)
```

Containers on the same bridge can talk to each other via their IPs. Docker adds iptables rules to NAT outbound traffic from containers through the host's physical NIC. Incoming traffic on published ports (`-p`) is forwarded from the host port to the container port via iptables DNAT.

## 2. The `bridge` Driver

The default network driver. Every container that doesn't specify a network joins the default `bridge` network.

### Default bridge (`docker0`)

The bridge named `bridge` exists by default. Containers on it can reach each other by IP, but **not by name** — Docker does not provide DNS on the default bridge.

```bash
# These two containers share the default bridge but cannot resolve each other by name
docker run -d --name web  nginx:alpine
docker run -d --name db   postgres:16

# web cannot reach db by hostname — only by IP (which changes on restart)
docker exec web ping db   # FAILS: name not resolved
```

**The legacy `--link` flag** provided name resolution on the default bridge but is deprecated. It was a hack — DNS via `/etc/hosts` editing — not real service discovery.

### User-defined bridge networks

When you create your own bridge network, Docker provides **automatic DNS resolution by container name**. This is the correct solution.

```bash
docker network create mynet
docker run -d --name web --network mynet nginx:alpine
docker run -d --name db  --network mynet postgres:16

# Now web CAN reach db by name
docker exec web ping db   # works
```

User-defined bridges also provide better isolation — containers on different user-defined networks cannot communicate unless explicitly connected.

In [ ]:
# Create a user-defined bridge and demonstrate DNS resolution
!docker network create demo-net

!docker run -d --name server --network demo-net nginx:alpine
!docker run -d --name client --network demo-net alpine sleep 60

import time; time.sleep(1)

# DNS resolution by container name
print("Resolving 'server' from 'client':")
!docker exec client nslookup server 2>/dev/null | grep -E 'Name|Address' | head -4

# Actual HTTP request by name
print("\nHTTP request using container name:")
!docker exec client wget -qO- http://server 2>/dev/null | head -3

In [ ]:
# Containers on different networks cannot reach each other
!docker network create isolated-net
!docker run -d --name outsider --network isolated-net alpine sleep 60

print("Ping from isolated network to demo-net container:")
!docker exec outsider ping -c 1 -W 2 server 2>&1 || echo "Cannot reach 'server' — different network"

# Clean up
!docker rm -f server client outsider
!docker network rm demo-net isolated-net

## 3. The `host` Driver

With the `host` driver, the container shares the host's network namespace entirely. There is no virtual interface, no NAT, no port mapping — the container's process binds directly to the host's network interfaces.

```bash
docker run --network host nginx
# Nginx listens on host port 80 directly — no -p needed
```

### When to use `host`

- **Maximum network performance** — no NAT overhead, useful for high-throughput or low-latency workloads (network appliances, monitoring agents)
- **When you need to bind to all host interfaces** without port mapping complexity
- **Monitoring containers** that need to inspect host network traffic

### Limitations

- **Linux only** — on macOS and Windows, Docker runs in a VM so `host` means the VM's network, not the real host
- **No isolation** — port conflicts with other host processes are possible
- **Security tradeoff** — the container can see all host network traffic

In [ ]:
# host network: container sees the same IP as the host
import subprocess

# Host's network interfaces
host_ip = subprocess.check_output(
    "hostname -I | awk '{print $1}'", shell=True
).decode().strip()
print(f"Host IP: {host_ip}")

# Container with host network sees the same IP
print("Container (host network) IP:")
!docker run --rm --network host alpine \
    sh -c 'hostname -I | awk "{print \$1}"'

## 4. The `none` Driver

The `none` driver gives the container a network namespace with only a loopback interface (`lo`). No external connectivity — it cannot reach the host, other containers, or the internet.

```bash
docker run --network none myapp
```

### Use cases

- **Batch processing jobs** that only read from / write to mounted volumes — no network needed
- **Security-sensitive workloads** that should have zero network exposure
- **Testing** network-failure scenarios
- **Sandboxing untrusted code** — the container cannot exfiltrate data

In [ ]:
# none network: only loopback exists
print("Network interfaces in 'none' container:")
!docker run --rm --network none alpine ip link show

print("\nTrying to reach the internet:")
!docker run --rm --network none alpine \
    ping -c 1 -W 2 8.8.8.8 2>&1 || echo "No route to host — as expected"

## 5. The `overlay` Driver

The `overlay` driver creates a distributed network that spans multiple Docker hosts. Containers on different physical machines can communicate as if they were on the same LAN.

```
Host A                          Host B
├── container1 (10.0.0.2)      ├── container3 (10.0.0.4)
└── container2 (10.0.0.3)      └── container4 (10.0.0.5)
         └──────── overlay network (10.0.0.0/24) ────────┘
            (VXLAN encapsulation over host network)
```

Overlay networks require Docker **Swarm mode** to be initialised — Swarm provides the key-value store that synchronises network state across hosts.

```bash
# Initialise Swarm on the first host
docker swarm init

# Create an overlay network
docker network create --driver overlay my-overlay

# Deploy a service that uses it
docker service create --network my-overlay --name web nginx:alpine
```

**VXLAN encapsulation:** overlay packets are wrapped in UDP packets and sent over the host network. The overhead is small but measurable — use `host` networking for maximum throughput when all containers are on the same host.

In Kubernetes, CNI plugins (Flannel, Calico, Cilium) implement similar overlay or BGP-based networking — the concept is the same.

## 6. The `macvlan` Driver

The `macvlan` driver assigns each container its own MAC address and connects it directly to the physical network — the container appears as a physical device on the LAN with its own IP.

```bash
docker network create \
    --driver macvlan \
    --subnet 192.168.1.0/24 \
    --gateway 192.168.1.1 \
    --opt parent=eth0 \
    macvlan-net

docker run --network macvlan-net \
    --ip 192.168.1.50 \
    nginx:alpine
# This container is reachable at 192.168.1.50 from the LAN
```

### When to use `macvlan`

- **Legacy applications** that expect to be reached directly on the LAN (not via NAT or port mapping)
- **Network appliances** that need a real MAC/IP on the physical network
- When you need containers addressable at **specific IPs** without router configuration

### Limitations

- Requires the physical NIC to be in **promiscuous mode** — many cloud providers don't allow this
- The container **cannot communicate with the host** by default (macvlan isolation)
- Not suitable for most general-purpose web applications

## 7. Inspecting Networks

In [ ]:
# List all networks on this Docker host
!docker network ls

In [ ]:
import subprocess, json

# Create and inspect a user-defined bridge
!docker network create --subnet 172.30.0.0/24 inspect-demo

raw = subprocess.check_output(["docker", "network", "inspect", "inspect-demo"])
data = json.loads(raw)[0]

print(f"Name:    {data['Name']}")
print(f"Driver:  {data['Driver']}")
print(f"Scope:   {data['Scope']}")
ipam = data['IPAM']['Config']
if ipam:
    print(f"Subnet:  {ipam[0].get('Subnet', 'auto')}")
    print(f"Gateway: {ipam[0].get('Gateway', 'auto')}")

!docker network rm inspect-demo

In [ ]:
# See which network(s) a container is connected to
!docker network create net-a
!docker network create net-b
!docker run -d --name multi-net --network net-a alpine sleep 60

# Connect the same container to a second network
!docker network connect net-b multi-net

raw = subprocess.check_output(["docker", "inspect", "multi-net",
                               "--format", "{{json .NetworkSettings.Networks}}"])
nets = json.loads(raw)
print("Networks this container is on:")
for net_name, cfg in nets.items():
    print(f"  {net_name}: {cfg['IPAddress']}")

!docker rm -f multi-net
!docker network rm net-a net-b

## 8. Network Driver Comparison

| Driver | Scope | DNS | Isolation | Use case |
|--------|-------|-----|-----------|----------|
| `bridge` (default) | Single host | No (by name) | Medium | Legacy, quick demos |
| `bridge` (user-defined) | Single host | **Yes** | Good | Standard multi-container apps |
| `host` | Single host | Host's DNS | None | Max performance, monitoring |
| `none` | Single host | None | **Total** | Batch jobs, sandboxing |
| `overlay` | Multi-host | Yes | Good | Swarm services, multi-host |
| `macvlan` | Physical LAN | External DNS | LAN-level | Legacy apps, network appliances |

### Decision flowchart

```
Multiple Docker hosts?
  → Yes → overlay (Swarm) or Kubernetes CNI
  → No ↓

Need full network isolation (no connectivity)?
  → Yes → none
  → No ↓

Need maximum performance / host-level access?
  → Yes → host
  → No ↓

Need containers on the physical LAN with real IPs?
  → Yes → macvlan
  → No ↓

→ User-defined bridge  ← the right default for most applications
```

## Summary

- Docker containers have their own network namespace. Network drivers determine how that namespace is connected to other containers and the outside world.
- **Default bridge** (`docker0`): containers talk by IP only — no DNS resolution by name. Use only for quick experiments.
- **User-defined bridge**: automatic DNS resolution by container name. Good isolation between networks. The correct choice for most multi-container applications on a single host.
- **`host`**: container shares the host's network stack. No NAT, no port mapping — maximum performance. Linux only, no isolation.
- **`none`**: loopback only. Zero external connectivity. For batch jobs, sandboxing, and security-sensitive workloads.
- **`overlay`**: spans multiple Docker hosts via VXLAN. Requires Swarm mode. Used by Docker Swarm services.
- **`macvlan`**: gives containers real MAC/IP addresses on the physical LAN. For legacy apps and network appliances. Requires promiscuous mode on the NIC.
- `docker network connect` attaches a running container to an additional network — a container can be on multiple networks simultaneously.